# **Stage 1 — Macro Variable Forecasting**
## SARIMA Pipeline | US Quarterly
___
**Purpose:** Forecast the 12 raw macroeconomic series from the EDA analytical
dataset, reconstruct derived variables from those forecasts, and assemble the
complete historical + forecast regressor dataset used as input for Stage 2
PD modelling.

**Data source:** Reads directly from the project GitHub repository — no local
paths required. Outputs are saved to the local `Data Collection/` folder and
should be committed to the repository after running.

| Step | Description | Output |
|------|-------------|--------|
| 0 | Configuration & data loading | — |
| 1 | SARIMA pipeline functions | — |
| 2 | Run SARIMA on all 12 raw series | — |
| 3 | Derive transformed variables from forecasts | — |
| 4 | Assemble complete regressor dataset | `SARIMA_regressors_US_Q.csv` |
| 5 | Forecast summary & validation | — |

**Repository:** `hogandan85/ST-498` · branch `main`

**References:**
Bank & Eder (2021). *A Review on the Probability of Default for IFRS 9.* SSRN 3981339.
Djeundje & Crook (2019). Dynamic survival models with varying coefficients for credit risks.
*European Journal of Operational Research*, 275(1), 319–333.

## **0: Configuration & Data Loading**
___
All user-facing settings live here. To change the variable selection, forecast
horizon, lag structure, or output filenames — edit this cell only. The pipeline
functions in Section 1 are never modified directly.

**Data is loaded directly from the GitHub raw URL** — the repo must be public
or a personal access token must be set in `GITHUB_TOKEN`. No local path
configuration is needed to run this notebook on any machine.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.special import inv_boxcox
from sklearn.preprocessing import PowerTransformer
from pmdarima import auto_arima
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ── GitHub data source ────────────────────────────────────────────────────────
GITHUB_RAW_BASE = 'https://raw.githubusercontent.com/hogandan85/ST-498/main/Data%20Collection'
GITHUB_TOKEN    = None

# ── Local output directory ────────────────────────────────────────────────────
LOCAL_OUTPUT = Path.cwd()/ 'Data Collection'

# ── Forecast horizon ──────────────────────────────────────────────────────────
# 20 quarters = 5 years, consistent with IFRS 9 lifetime ECL horizon
N_PERIODS = 20
M         = 4

# ── Raw series to forecast ────────────────────────────────────────────────────
# Level/index series only. Derived variables are reconstructed from these
# forecasts in Section 3 — do not add derived cols here.
RAW_TO_FORECAST = [
    'us_real_gdp',
    'us_unemployment',
    'us_cpi',
    'us_consumer_confidence',
    'us_bond_yield_10y',
    'us_credit',
    'us_sp500_close',
    'us_vix',
    'us_house_price_idx',
    'us_industrial_production',
    'us_oil_price',
    'us_reer',
]

# ── Derived variable construction ─────────────────────────────────────────────
# Each entry: output_col -> (source_col, method, periods)
# method: 'pct_change_yoy' | 'pct_change_qoq' | 'diff' | 'log_diff'
DERIVED_VARS = {
    'us_gdp_yoy_growth':    ('us_real_gdp',              'pct_change_yoy', 4),
    'us_house_price_yoy':   ('us_house_price_idx',       'pct_change_yoy', 4),
    'us_indprod_yoy':       ('us_industrial_production',  'pct_change_yoy', 4),
    'us_oil_yoy':           ('us_oil_price',             'pct_change_yoy', 4),
    'us_reer_diff':         ('us_reer',                  'diff',           1),
    'us_credit_qoq_growth': ('us_credit',                'pct_change_qoq', 1),
    'us_sp500_log_ret':     ('us_sp500_close',           'log_diff',       1),
    'us_vix_log_ret':       ('us_vix',                   'log_diff',       1),
    'us_bond_yield_10y_d1': ('us_bond_yield_10y',        'diff',           1),
}

# ── Lag specification ─────────────────────────────────────────────────────────
# CCF-optimal lags from EDA Section 5
# Fast-transmitting variables: max 4Q window
# Slow-transmitting variables (house prices, credit, CPI): max 6Q window
LAG_SPEC = {
    'us_house_price_yoy':     3,   
    'us_consumer_confidence': 2,   
    'us_unemployment':        1,  
    'us_credit_qoq_growth':   6,   
    'us_gdp_yoy_growth':      0,   
    'us_cpi':                 6,   
    'us_bond_yield_10y_d1':   2,
    'us_sp500_log_ret':       4,
    'us_reer_diff':           1,
    'us_indprod_yoy':         3,
    'us_oil_yoy':             2,
    'us_vix_log_ret':         0,
}

# ── Output filenames ──────────────────────────────────────────────────────────
# When adding SARIMAX or ML models, change the prefix only.
MODEL_PREFIX       = 'SARIMA'
OUT_REGRESSORS = f'{MODEL_PREFIX}_regressors_US_Q.csv'

# ── Colour palette (Plotly) ───────────────────────────────────────────────────
NAVY, BLUE, RED, AMBER, GREEN, GREY = (
    '#1F3864', '#2E75B6', '#C0392B', '#E67E22', '#27AE60', '#7F8C8D')
TEMPLATE = 'plotly_white'
RECESSIONS_US = [
    ('1990-07-01', '1991-03-31', 'Early 1990s'),
    ('2001-03-01', '2001-12-31', 'Dot-com'),
    ('2007-12-01', '2009-06-30', 'GFC'),
    ('2020-02-01', '2020-04-30', 'COVID'),
]

# ── Load EDA analytical dataset from GitHub ───────────────────────────────────
def load_github_csv(filename: str, token: str = None) -> pd.DataFrame:
    url = f'{GITHUB_RAW_BASE}/{filename}'
    if token:
        import requests, io
        headers = {'Authorization': f'token {token}'}
        r = requests.get(url, headers=headers)
        r.raise_for_status()
        return pd.read_csv(io.StringIO(r.text), index_col=0, parse_dates=True)
    return pd.read_csv(url, index_col=0, parse_dates=True)

df_hist = load_github_csv('EDA_analytical_US_Q.csv', token=GITHUB_TOKEN)

print(f'Loaded EDA_analytical_US_Q.csv from GitHub')
print(f'  Shape  : {df_hist.shape}')
print(f'  From   : {df_hist.index.min().date()}  to  {df_hist.index.max().date()}')
print(f'  Target : us_delinquency_rate — '
      f'{df_hist["us_delinquency_rate"].notna().sum()} obs | '
      f'min {df_hist["us_delinquency_rate"].min():.2f}% | '
      f'max {df_hist["us_delinquency_rate"].max():.2f}% | '
      f'mean {df_hist["us_delinquency_rate"].mean():.2f}%')
print(f'\nRaw series to forecast ({len(RAW_TO_FORECAST)}):')
for col in RAW_TO_FORECAST:
    status = 'OK' if col in df_hist.columns else 'MISSING'
    nans   = df_hist[col].isna().sum() if col in df_hist.columns else '—'
    print(f'  [{status}]  {col:<35s}  NaN={nans}')

## **1: SARIMA Pipeline Functions**
___
Core pipeline functions. These are never called directly — they are invoked
by `run_sarima_pipeline()` in Section 2.

**Design note:** The transformation and fitting logic is kept separate from
the dataset assembly logic so that other forecasting approaches (SARIMAX, ML)
can plug into the same assembly pipeline in Section 3–4 by returning a
`results` dict in the same format.

In [ ]:
# ── Transformation helpers ────────────────────────────────────────────────────

def apply_transformations(series: pd.Series) -> dict:
    """Returns dict of {name: (transformed_series, inverse_fn)}."""
    transformations = {}
    has_negatives = (series <= 0).any()

    if not has_negatives:
        log_series = np.log(series)
        transformations['log'] = (log_series, lambda x: np.exp(x))

    if not has_negatives:
        bc_transformed, lam = stats.boxcox(series)
        bc_series = pd.Series(bc_transformed, index=series.index)
        transformations['boxcox'] = (bc_series, lambda x, l=lam: inv_boxcox(x, l))

    pt = PowerTransformer(method='yeo-johnson')
    yj_vals = pt.fit_transform((series + 0.0001).values.reshape(-1, 1)).flatten()
    yj_series = pd.Series(yj_vals, index=series.index)
    transformations['yeo-johnson'] = (
        yj_series,
        lambda x: pt.inverse_transform(np.array(x).reshape(-1, 1)).flatten() - 0.0001
    )
    return transformations


def extract_diagnostics(model) -> dict:
    """Extracts Ljung-Box, Jarque-Bera, heteroskedasticity stats from fitted model."""
    try:
        sarimax_model = model.arima_res_
        diag = {
            'ljung_box_q':    sarimax_model.test_serial_correlation('ljungbox', lags=1)[0][1][0],
            'jarque_bera_jb': sarimax_model.test_normality('jarquebera')[0][1],
            'heterosked_h':   sarimax_model.test_heteroskedasticity('breakvar')[0][1],
        }
        resid = sarimax_model.resid
        from scipy.stats import skew, kurtosis
        return {
            'prob_q':   round(diag['ljung_box_q'], 4),
            'prob_jb':  round(diag['jarque_bera_jb'], 4),
            'prob_h':   round(diag['heterosked_h'], 4),
            'skew':     round(float(skew(resid)), 4),
            'kurtosis': round(float(kurtosis(resid, fisher=False)), 4),
        }
    except Exception:
        return {'prob_q': None, 'prob_jb': None, 'prob_h': None,
                'skew': None, 'kurtosis': None}


# ── Model fitting ─────────────────────────────────────────────────────────────

def fit_all_transformations(series: pd.Series, m: int = 4) -> pd.DataFrame:
    """
    Fits auto_arima for each eligible transformation.
    Returns a comparison DataFrame sorted by AIC (best first).
    """
    transformations = apply_transformations(series)
    results = []

    for name, (transformed, inverse_fn) in transformations.items():
        try:
            model = auto_arima(
                y=transformed,
                start_p=0, start_q=0, max_p=3, max_q=3,
                start_P=0, start_Q=0, max_P=2, max_Q=2,
                m=m, d=None, D=None, seasonal=True,
                information_criterion='aic',
                error_action='ignore', suppress_warnings=True, stepwise=True
            )
            residuals   = pd.Series(model.resid())
            diagnostics = extract_diagnostics(model)
            results.append({
                'transformation': name,
                'aic':            model.aic(),
                'bic':            model.bic(),
                'order':          model.order,
                'seasonal_order': model.seasonal_order,
                'residual_std':   residuals.std(),
                'abs_skew':       abs(residuals.skew()),
                'abs_kurt':       abs(residuals.kurtosis() - 3),
                'prob_q':         diagnostics['prob_q'],
                'prob_jb':        diagnostics['prob_jb'],
                'prob_h':         diagnostics['prob_h'],
                'skew':           diagnostics['skew'],
                'kurtosis':       diagnostics['kurtosis'],
                '_model':         model,
                '_inverse_fn':    inverse_fn,
            })
        except Exception as e:
            print(f'    ⚠ Failed [{name}]: {e}')

    return pd.DataFrame(results).sort_values('aic').reset_index(drop=True)


def forecast_best_model(comparison_df: pd.DataFrame,
                         n_periods: int = 20) -> dict:
    """Takes comparison df, selects best model by AIC, returns forecast."""
    best       = comparison_df.iloc[0]
    model      = best['_model']
    inverse_fn = best['_inverse_fn']
    return {
        'transformation': best['transformation'],
        'order':          best['order'],
        'seasonal_order': best['seasonal_order'],
        'aic':            best['aic'],
        'forecast':       inverse_fn(model.predict(n_periods=n_periods)),
        'model':          model,
        'inverse_fn':     inverse_fn,
    }


# ── Main pipeline ─────────────────────────────────────────────────────────────

def run_sarima_pipeline(df: pd.DataFrame, m: int = 4,
                         n_periods: int = 20) -> tuple:
    """
    Runs the full SARIMA pipeline for every column in df.

    Returns
    -------
    all_results     : dict {col: result_dict}  — use for dataset assembly
    all_comparisons : dict {col: comparison_df} — use for model diagnostics
    """
    all_results, all_comparisons = {}, {}

    for col in df.columns:
        print(f"\n{'='*60}\n  Processing: {col}\n{'='*60}")
        series        = df[col].dropna()
        comparison_df = fit_all_transformations(series, m=m)

        display_cols = ['transformation', 'aic', 'bic', 'order', 'seasonal_order',
                        'residual_std', 'prob_q', 'prob_jb', 'prob_h', 'skew', 'kurtosis']
        print(comparison_df[display_cols].to_string(index=False))

        result = forecast_best_model(comparison_df, n_periods=n_periods)
        print(f"\n  Best: [{result['transformation']}]  "
              f"SARIMA{result['order']}x{result['seasonal_order']}  "
              f"AIC={result['aic']:.2f}")

        all_results[col]     = result
        all_comparisons[col] = comparison_df

    return all_results, all_comparisons

## **2: Run SARIMA Forecasts**
___
Runs the SARIMA pipeline on all 12 raw series defined in `RAW_TO_FORECAST`.
For each series, three transformations (log, Box-Cox, Yeo-Johnson) are fitted
and compared by AIC. The best transformation is selected and used to generate
a 20-quarter forecast (2026 Q1 – 2030 Q4).

**Runtime:** approximately 5–20 minutes depending on machine.

**Output:** `results` dict — `{column: {'forecast': array, 'order': ..., ...}}`
This dict is consumed by Section 3. If you want to swap SARIMA for another
model, replace this cell with your model's fitting code and ensure `results`
has the same structure.

In [ ]:
# ── COVID exclusion (drop-in-place, for simplicity) ───────────────────────────
# Remove the COVID window from the series used to FIT the SARIMA models. The
# 2020Q2 unemployment spike (11.0%) and GDP collapse are extreme outliers that
# distort the AR + seasonal estimates; this is the simple route for the US PoC.
#
# CAVEAT (document in the write-up): removing *rows* and re-joining collapses the
# ~2-year hole into a single quarter step and shifts the quarterly seasonal phase
# across the join, so this is NOT identical to a Salkever (1976) in-place impulse
# dummy (which preserves the calendar). The calendar is restored in Section 4
# (COVID quarters reinserted as NaN) so every lag/growth column stays
# calendar-correct for Stage 2. For a fully defensible treatment, fit on a
# NaN-in-place series via statsmodels SARIMAX instead of dropping (see notes).

# Closed interval [start, end] — BOTH endpoints are dropped. Edit here only.
COVID_DROP_START = '2020-03-31'   # 2020 Q1
COVID_DROP_END   = '2021-12-31'   # 2021 Q4  (set to '2021-09-30' to KEEP 2021 Q4)

_covid_mask = (df_hist.index >= COVID_DROP_START) & (df_hist.index <= COVID_DROP_END)
_dropped    = df_hist.index[_covid_mask]
df_hist     = df_hist.loc[~_covid_mask].sort_index()   # idempotent: safe to re-run

print(f'COVID window dropped: {len(_dropped)} quarters '
      f'({COVID_DROP_START} → {COVID_DROP_END})')
print('  ' + (', '.join(f'{d.year}Q{d.quarter}' for d in _dropped) or '(none)'))
print(f'  df_hist now: {df_hist.shape[0]} rows | '
      f'{df_hist.index.min().date()} → {df_hist.index.max().date()}')

In [ ]:
df_hist

In [ ]:
results, comparisons = run_sarima_pipeline(
    df_hist[RAW_TO_FORECAST],
    m=M,
    n_periods=N_PERIODS
)

print(f'\nCompleted SARIMA fitting for {len(results)} series.')
print('\nModel summary:')
print(f'  {"Series":<35s} {"Transform":<12s} {"Order":<15s} {"Seasonal":<20s} {"AIC":>8s}')
print(f'  {"-"*90}')
for col, res in results.items():
    print(f'  {col:<35s} {res["transformation"]:<12s} '
          f'{str(res["order"]):<15s} {str(res["seasonal_order"]):<20s} '
          f'{res["aic"]:>8.2f}')

## **3: Derive Transformed Variables**
___
Reconstructs derived variables (YoY growth rates, log returns, first
differences) from the raw SARIMA forecasts. Derived variables require
historical context for the YoY/QoQ calculations — the historical raw series
is concatenated with the forecast period before computing each transformation.

**Why derive rather than forecast directly:**
Forecasting `us_gdp_yoy_growth` directly with SARIMA would treat it as an
independent series, ignoring its definitional relationship with `us_real_gdp`.
Deriving it from the GDP forecast preserves that relationship and ensures
internal consistency between the raw and derived forecasts.

**Output:** Derived variables are appended directly to `forecast_raw` in
memory and passed to Section 4 for assembly into the regressor dataset.

In [ ]:
# ── Build forecast date index ─────────────────────────────────────────────────
last_date    = df_hist.index[-1]
forecast_idx = pd.date_range(
    start=last_date + pd.DateOffset(months=3),
    periods=N_PERIODS,
    freq='QE'
)

# ── Assemble raw forecasts ────────────────────────────────────────────────────
forecast_raw = pd.DataFrame(index=forecast_idx)
for col in RAW_TO_FORECAST:
    if col in results:
        # np.asarray() -> positional assignment. Dropping COVID rows kills the
        # index frequency, so the fitted model returns forecasts on an INTEGER
        # index; assigning the raw Series would align on index -> silent all-NaN.
        forecast_raw[col] = np.asarray(results[col]['forecast'])

# ── Reconstruct derived variables ─────────────────────────────────────────────
# Concatenate historical raw series with forecast period so that YoY/QoQ
combined = pd.concat([df_hist[RAW_TO_FORECAST], forecast_raw])

def build_derived(combined, source_col, method, periods, forecast_idx):
    """Compute a derived variable and slice out the forecast period."""
    if method == 'pct_change_yoy':
        return (combined[source_col].pct_change(periods) * 100).loc[forecast_idx]
    elif method == 'pct_change_qoq':
        return (combined[source_col].pct_change(periods) * 100).loc[forecast_idx]
    elif method == 'diff':
        return combined[source_col].diff(periods).loc[forecast_idx]
    elif method == 'log_diff':
        return np.log(combined[source_col]).diff(periods).loc[forecast_idx]
    else:
        raise ValueError(f'Unknown method: {method}')

for out_col, (src_col, method, periods) in DERIVED_VARS.items():
    forecast_raw[out_col] = build_derived(combined, src_col, method, periods, forecast_idx)

# ── Preview derived variables ────────────────────────────────────────────────
print(f'Derived variables constructed: {list(DERIVED_VARS.keys())}')
print(f'forecast_raw shape: {forecast_raw.shape}')
nan_counts = forecast_raw.isna().sum()
nan_cols   = nan_counts[nan_counts > 0]
if len(nan_cols) == 0:
    print('  No NaN values.')
else:
    print(f'  NaN values: {nan_cols.to_dict()}')

# ── Plot: all raw forecasts ───────────────────────────────────────────────────
n_cols = 3
n_rows = int(np.ceil(len(RAW_TO_FORECAST) / n_cols))

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=[c.replace('us_', '').replace('_', ' ').title()
                    for c in RAW_TO_FORECAST],
    vertical_spacing=0.08, horizontal_spacing=0.06
)

for i, col in enumerate(RAW_TO_FORECAST):
    r, c = i // n_cols + 1, i % n_cols + 1
    hist_series = df_hist[col].dropna()
    fc_vals     = forecast_raw[col]

    fig.add_trace(go.Scatter(
        x=hist_series.index[-40:], y=hist_series.values[-40:],
        mode='lines', line=dict(color=NAVY, width=1.5),
        showlegend=(i == 0), name='Historical'), row=r, col=c)

    fig.add_trace(go.Scatter(
        x=fc_vals.index, y=fc_vals.values,
        mode='lines', line=dict(color=RED, width=1.5, dash='dash'),
        showlegend=(i == 0), name='SARIMA Forecast'), row=r, col=c)

fig.update_layout(
    title=dict(
        text='Figure 1 — SARIMA Forecasts: All Raw Series (2026–2030)<br>'
             '<sup>Last 10 years of history shown | Red dashed = forecast</sup>',
        font=dict(size=13, color=NAVY)),
    template=TEMPLATE, height=n_rows * 250,
    margin=dict(t=80, b=40, l=40, r=20))
fig.update_annotations(font_size=9)
fig.show()

## **4: Assemble Complete Regressor Dataset**
___
Concatenates the full historical EDA dataset with the forecast-period raw
series, then recomputes all lagged columns on the combined series.

Lagged columns must be recomputed on the combined series — not pre-computed
separately for history and forecast — so that lag shifts correctly bridge the
boundary. For example, `us_house_price_yoy_L3` at 2026-Q1 is the value of
`us_house_price_yoy` at 2025-Q2, which is in the historical data.

**Lag structure from EDA Section 5 (Bank & Eder 2021, Section 10;**
**Djeundje & Crook 2019):**

| Variable | Lag | Transmission mechanism |
|---|---|---|
| `us_house_price_yoy` | L3 | Mortgage stress: 6–18 months |
| `us_consumer_confidence` | L2 | Leading indicator |
| `us_unemployment` | L1 | Lagging but within 1 year |
| `us_credit_qoq_growth` | L6 | Leverage build-up: 1–2 years |
| `us_gdp_yoy_growth` | L0 | Contemporaneous |
| `us_cpi` | L6 | Real income erosion |

**Output:** `SARIMA_regressors_US_Q.csv` — 164 rows × 42 columns saved to
`Data Collection/` locally and committed to the repository. This is the
single output of this notebook and the direct input for `Stage2_PD_Modelling.ipynb`.

In [ ]:
# ── Concatenate historical + forecast ────────────────────────────────────────
# All columns retained — NaN in forecast rows for hist-only columns
combined_full = pd.concat([df_hist, forecast_raw], axis=0).sort_index()

# ── Restore a continuous quarterly calendar ───────────────────────────────────
# COVID quarters were dropped only for model FITTING. Reinsert them here as NaN
# so the index is gap-free and every .shift()/.diff() below is computed on true
# calendar spacing. A positional shift across the hole would mislabel the lags
# for quarters adjacent to the gap. NaN transparently marks excluded quarters.
combined_full = combined_full.asfreq('QE-DEC')

# ── Recompute lagged columns on combined series ───────────────────────────────
for var, lag in LAG_SPEC.items():
    if var in combined_full.columns:
        combined_full[f'{var}_L{lag}'] = combined_full[var].shift(lag)
    else:
        print(f'{var} not found in combined dataset — lag column skipped')

# ── Save ──────────────────────────────────────────────────────────────────────
out_path = LOCAL_OUTPUT / OUT_REGRESSORS
combined_full.to_csv(out_path)
print(f'Saved {OUT_REGRESSORS}  {combined_full.shape}')
print(f'  Path            : {out_path}')
n_reinserted = len(combined_full) - len(df_hist) - len(forecast_raw)
print(f'  Historical rows (fitted)    : {len(df_hist)}')
print(f'  COVID rows reinserted (NaN) : {n_reinserted}')
print(f'  Forecast rows               : {len(forecast_raw)}')
print(f'  Total rows                  : {len(combined_full)}')
print(f'  Total columns   : {combined_full.shape[1]}')

# ── Validate: show forecast period lagged regressors ─────────────────────────
sig_lag_cols = [f"{v}_L{l}" for v, l in LAG_SPEC.items()
                if v in ['us_house_price_yoy', 'us_consumer_confidence',
                         'us_unemployment', 'us_credit_qoq_growth',
                         'us_gdp_yoy_growth', 'us_cpi']]
print('\nForecast period — Stage 2 significant regressors:')
print(combined_full.loc[forecast_idx, sig_lag_cols].round(4).to_string())

nan_check = combined_full.loc[forecast_idx, sig_lag_cols].isna().sum()
if nan_check.any():
    print('\nNaN values in significant regressors during forecast period:')
    print(nan_check[nan_check > 0])
else:
    print('\n✓ No NaN values in significant regressors during forecast period.')

## **5: Forecast Summary & Validation**
___
Sanity checks and a summary plot of the six significant predictors from the
CCF analysis. These are the series that enter the Stage 2 PD regression at
their optimal lags — validating their forecast behaviour is a prerequisite
before running the modelling notebook.

**Key things to check:**
- Forecasts are economically plausible (no explosive paths)
- No flat lines at the last observed value (would indicate SARIMA(0,1,0)
  random walk — acceptable but noted)
- No NaN values in the significant regressor columns

In [ ]:
# ── Significant predictors forecast plot ──────────────────────────────────────
SIG_VARS = {
    'us_house_price_yoy':     'House Price YoY (%)',
    'us_consumer_confidence': 'Consumer Confidence',
    'us_unemployment':        'Unemployment (%)',
    'us_credit_qoq_growth':   'Credit QoQ Growth (%)',
    'us_gdp_yoy_growth':      'GDP YoY Growth (%)',
    'us_cpi':                 'CPI',
}

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=list(SIG_VARS.values()),
    vertical_spacing=0.12, horizontal_spacing=0.07
)

for i, (var, label) in enumerate(SIG_VARS.items()):
    r, c = i // 3 + 1, i % 3 + 1

    hist_series = df_hist[var].dropna()
    fc_series   = forecast_raw[var] if var in forecast_raw.columns else None

    # Last 10 years of history
    cutoff = hist_series.index[-1] - pd.DateOffset(years=10)
    hist_plot = hist_series[hist_series.index >= cutoff]

    for s, e, lbl in RECESSIONS_US:
        if pd.Timestamp(s) >= cutoff:
            fig.add_vrect(x0=s, x1=e, fillcolor='lightgrey', opacity=0.3,
                          layer='below', line_width=0, row=r, col=c)

    fig.add_trace(go.Scatter(
        x=hist_plot.index, y=hist_plot.values,
        mode='lines', line=dict(color=NAVY, width=1.5),
        showlegend=(i == 0), name='Historical'), row=r, col=c)

    if fc_series is not None:
        fig.add_trace(go.Scatter(
            x=fc_series.index, y=fc_series.values,
            mode='lines', line=dict(color=RED, width=1.8, dash='dash'),
            showlegend=(i == 0), name='SARIMA Forecast'), row=r, col=c)

fig.update_layout(
    title=dict(
        text='Figure 2 — SARIMA Forecasts: CCF-Significant Predictors (2026–2030)<br>'
             '<sup>Last 10 years of history | Grey = recessions | '
             'These series enter Stage 2 at their CCF-optimal lags</sup>',
        font=dict(size=13, color=NAVY)),
    template=TEMPLATE, height=500,
    legend=dict(orientation='h', yanchor='bottom', y=1.02,
                xanchor='right', x=1, font=dict(size=10)),
    margin=dict(t=100, b=40, l=50, r=20))
fig.update_annotations(font_size=10)
fig.show()

# ── Summary table ─────────────────────────────────────────────────────────────
print('\nForecast Summary — Significant Predictors:')
print(f'  {"Variable":<35s} {"Hist last":>10s} {"FC mean":>10s} '
      f'{"FC min":>10s} {"FC max":>10s} {"Flat?":>6s}')
print(f'  {"-"*80}')
for var in SIG_VARS:
    if var not in forecast_raw.columns:
        continue
    hist_last = df_hist[var].dropna().iloc[-1]
    fc        = forecast_raw[var].dropna()
    flat      = 'Yes' if fc.std() < 0.001 else 'no'
    print(f'  {var:<35s} {hist_last:>10.3f} {fc.mean():>10.3f} '
          f'{fc.min():>10.3f} {fc.max():>10.3f} {flat:>6s}')

print(f'  Output : {OUT_REGRESSORS}')

## **6: Out-of-Sample Validation — Recursive Backtest & Diebold-Mariano**
___
Replaces the old in-sample `[Transform, AIC, BIC]` table with the out-of-sample
report table **`[Variable, Winning model, RMSE, DM-validated]`**, plus two forecast
figures (all series; DM-validated series only).

**Why this needs a backtest.** RMSE and the Diebold-Mariano (DM) test are
*out-of-sample* quantities: DM tests whether the mean loss-differential between
two forecasts is zero over a **sample of forecast errors** (Diebold & Mariano
1995). Sections 2-5 fit each model once on the whole series and hold nothing out,
so there is no error sample to compute either from. This section runs a
**rolling-origin (expanding-window, recursive) backtest**: at each origin the
selected specification is re-estimated on data up to that point, an `h`-step
forecast is made, and its error recorded — producing the error sample RMSE and DM
require.

**Design choices (edit the config block below):**
- *Scheme:* expanding window (recursive). Maximises training data at each origin —
  the recommended default for short series (Petropoulos et al. 2022). Switch to
  `rolling` for robustness to structural breaks (Clark & McCracken 2009).
- *Horizon:* `h = 1`. One-step loss differentials are serially uncorrelated, giving
  the DM test its cleanest variance and maximum power; `h > 1` needs the HAC term
  and loses power fast at this sample size.
- *Benchmark:* naive random walk (last observed value) — the universal yardstick.
- *DM-validated flag:* a random walk is **nested** inside almost every ARIMA spec,
  and the standard DM test is **invalid for nested models** (its asymptotics
  collapse if the smaller model is true). The flag therefore uses the
  **Clark-West (2007)** adjustment, the correct nested-model test; the flag additionally requires the model to beat naive in realised RMSE (skill > 0) so it never contradicts the RMSE column. The
  HLN-corrected DM statistic (Harvey, Leybourne & Newbold 1997; Student-t with
  n-1 df) is also reported in the diagnostics for transparency.

**Caveats (state these in the write-up):**
- The model *order* is chosen once on the full sample (Section 2) and only
  *re-estimated* — not re-selected — at each origin. RMSE is therefore mildly
  optimistic (specification look-ahead). For a fully honest but far slower variant,
  re-run `auto_arima` inside the origin loop.
- At ~70 origins the DM/CW tests remain low-powered; read "DM-validated = No" as
  *"not shown to beat naive"*, not *"equal to naive"*.
- Origins whose `h`-step target crosses the dropped-COVID gap are excluded, so
  every evaluated step is a true `h`-quarter step.
- RMSE is in each variable's **native units** (not comparable across rows); use the diagnostics' skill-% and DM/CW columns for cross-variable comparison.


In [ ]:
# ── Out-of-sample validation: recursive backtest + Diebold-Mariano ─────────────
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy import stats, special

# ── Backtest configuration ─────────────────────────────────────────────────────
BACKTEST_H         = 1            # evaluation horizon (quarters). 1 => cleanest DM
BACKTEST_MIN_TRAIN = 60           # initial expanding-window length before 1st origin
BACKTEST_SCHEME    = 'expanding'  # 'expanding' (recursive) | 'rolling'
BACKTEST_ROLL_WIN  = 60           # window length when scheme == 'rolling'
DM_ALPHA           = 0.05         # significance level for the DM-validated flag
YOY_EVAL_START     = '2023-01-01' # derived-YoY eval starts 2023 Q1 (drop COVID year-ago bases)

def classify_model(order, seasonal_order):
    p, d, q = order; P, D, Q, m = seasonal_order
    if p == 0 and q == 0 and P == 0 and Q == 0: return 'Mean'
    if P > 0 or Q > 0 or D > 0:                  return 'SARMA'
    if q == 0 and Q == 0:                        return 'AR'
    return 'ARMA'

# ── Per-window transform (refit on TRAIN only — no look-ahead) ──────────────────
def _train_transform(train, name):
    if name == 'log':
        return np.log(train), (lambda x: np.exp(x))
    if name == 'boxcox':
        v, lam = stats.boxcox(train)
        return pd.Series(v, index=train.index), (lambda x, l=lam: special.inv_boxcox(x, l))
    if name == 'yeo-johnson':
        pt = PowerTransformer(method='yeo-johnson')
        v  = pt.fit_transform((train + 0.0001).values.reshape(-1, 1)).flatten()
        return (pd.Series(v, index=train.index),
                lambda x, p=pt: p.inverse_transform(np.array(x).reshape(-1, 1)).flatten() - 0.0001)
    return train, (lambda x: x)   # 'none'

# ── Recursive rolling-origin backtest for one series ────────────────────────────
def recursive_backtest(series, order, seasonal_order, tf_name,
                       h=BACKTEST_H, min_train=BACKTEST_MIN_TRAIN,
                       scheme=BACKTEST_SCHEME, roll_win=BACKTEST_ROLL_WIN):
    """Re-estimate the selected spec at each origin; return model & naive-RW
    error samples (calendar-correct, non-finite and gap-crossing steps skipped)."""
    s = series.dropna(); qi = s.index
    em, en, fm, fn, od = [], [], [], [], []
    start = min(min_train, max(40, len(s) - 24))
    for t in range(start, len(s) - h + 1):
        tr  = s.iloc[:t] if scheme == 'expanding' else s.iloc[max(0, t - roll_win):t]
        tgt = t + h - 1
        if (qi[tgt].to_period('Q') - qi[t - 1].to_period('Q')).n != h:   # gap guard
            continue
        try:
            ytr, inv = _train_transform(tr, tf_name)
            res = SARIMAX(ytr, order=order, seasonal_order=seasonal_order,
                          enforce_stationarity=False, enforce_invertibility=False
                          ).fit(disp=False, maxiter=50)
            fc = float(inv(res.forecast(h).iloc[-1]))
        except Exception:
            continue
        a = float(s.iloc[tgt]); nv = float(tr.iloc[-1])
        if not np.all(np.isfinite([fc, a, nv])):
            continue
        em.append(a - fc); en.append(a - nv); fm.append(fc); fn.append(nv); od.append(qi[tgt])
    return (np.array(em), np.array(en), np.array(fm), np.array(fn), pd.DatetimeIndex(od))

# ── Diebold-Mariano with HLN small-sample correction (HAC when h>1) ─────────────
def dm_hln(e_bench, e_model, h=BACKTEST_H):
    d = e_bench ** 2 - e_model ** 2; n = len(d)          # >0 => model beats benchmark
    if n < 3: return np.nan, np.nan
    var = np.var(d, ddof=0)
    for k in range(1, h):
        var += 2 * np.cov(d[:-k], d[k:])[0, 1]
    var /= n
    if var <= 0: return np.nan, np.nan                   # negative LRV -> DM undefined
    dm   = d.mean() / np.sqrt(var)
    corr = np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)   # Harvey-Leybourne-Newbold 1997
    dm_s = dm * corr
    return dm_s, 2 * (1 - stats.t.cdf(abs(dm_s), df=n - 1))

# ── Clark-West adjustment (correct test when benchmark is NESTED) ───────────────
def clark_west(e_bench, e_model, f_bench, f_model):
    ft = e_bench ** 2 - (e_model ** 2 - (f_bench - f_model) ** 2); n = len(ft)
    if n < 3: return np.nan, np.nan
    t = ft.mean() / (ft.std(ddof=1) / np.sqrt(n))
    return t, 1 - stats.norm.cdf(t)                       # one-sided: model better

# ── Ljung-Box on the OOS error sample (white-noise check on forecast errors) ────
def ljung_box_oos(err, m=M):
    """Ljung-Box Q on the OOS forecast errors. An optimal h-step forecast leaves
    white-noise errors; a low p-value flags autocorrelation the model failed to
    exploit. Lags = one seasonal cycle, capped for the sample size."""
    n = len(err)
    lag = int(min(2 * m, max(1, n // 5)))
    if n < lag + 2:
        return np.nan, np.nan, lag
    try:
        lb = acorr_ljungbox(np.asarray(err), lags=[lag], return_df=True)
        return float(lb['lb_stat'].iloc[-1]), float(lb['lb_pvalue'].iloc[-1]), lag
    except Exception:
        return np.nan, np.nan, lag

# ── Run the backtest for every forecast variable ───────────────────────────────
print(f'Recursive {BACKTEST_SCHEME}-window backtest | h={BACKTEST_H} | '
      f'benchmark = naive random walk | flag = Clark-West one-sided @ {DM_ALPHA:.0%}\n')

bt_rows, bt_errors = [], {}
for var in RAW_TO_FORECAST:
    if var not in results:
        continue
    res = results[var]
    order, sorder, tf = res['order'], res['seasonal_order'], res['transformation']
    em, en, fm, fn, od = recursive_backtest(df_hist[var], order, sorder, tf)
    if len(em) < 3:
        bt_rows.append({'Variable': var, 'Winning model': classify_model(order, sorder),
                        'RMSE': np.nan, 'DM-validated': 'n/a', '_skill': np.nan,
                        '_r2_oos': None, '_mae': None, '_mae_naive': None,
                        '_dm_star': None, '_dm_p': None, '_cw_stat': None, '_cw_p': None,
                        '_lb_lag': None, '_lb_stat': None, '_lb_p': None,
                        '_n_origins': len(em), '_validated': False})
        continue
    rmse, rmse_n = float(np.sqrt(np.mean(em ** 2))), float(np.sqrt(np.mean(en ** 2)))
    dm, p_dm = dm_hln(en, em)
    cw, p_cw = clark_west(en, em, fn, fm)
    # ── Comparability metrics for the TS-vs-ML head-to-head (added) ──────────────
    mae, mae_n = float(np.mean(np.abs(em))), float(np.mean(np.abs(en)))
    sse_n  = float(np.sum(en ** 2))
    r2_oos = float(1 - np.sum(em ** 2) / sse_n) if sse_n > 0 else np.nan  # 1 - MSE/MSE_naive (Campbell-Thompson OOS R^2)
    lb_stat, lb_p, lb_lag = ljung_box_oos(em)                            # white-noise check on OOS errors
    # 'validated' = model BEATS naive in OOS RMSE *and* the gain is significant.
    # (Clark-West alone can flag significance even when realised RMSE is worse,
    #  because it adds back the estimation-noise penalty; require skill>0 too so
    #  the flag never contradicts the RMSE column shown beside it.)
    beats = rmse < rmse_n
    validated = bool(beats and np.isfinite(p_cw) and p_cw < DM_ALPHA)
    bt_rows.append({
        'Variable': var, 'Winning model': classify_model(order, sorder),
        'RMSE': round(rmse, 4), 'DM-validated': 'Yes' if validated else 'No',
        '_skill': round(100 * (1 - rmse / rmse_n), 1) if rmse_n else np.nan,
        '_r2_oos':  round(r2_oos, 3) if np.isfinite(r2_oos) else None,
        '_mae':     round(mae, 4),
        '_mae_naive': round(mae_n, 4),
        '_dm_star': round(dm, 3) if np.isfinite(dm) else None,
        '_dm_p':   round(p_dm, 4) if np.isfinite(p_dm) else None,
        '_cw_stat': round(cw, 3) if np.isfinite(cw) else None,
        '_cw_p':   round(p_cw, 4) if np.isfinite(p_cw) else None,
        '_lb_lag':  lb_lag,
        '_lb_stat': round(lb_stat, 3) if np.isfinite(lb_stat) else None,
        '_lb_p':    round(lb_p, 4) if np.isfinite(lb_p) else None,
        '_n_origins': len(em), '_validated': validated,
    })
    bt_errors[var] = dict(e_model=em, e_naive=en, f_model=fm, f_naive=fn, dates=od)

backtest_summary = pd.DataFrame(bt_rows)

# ── The report table: Variable | Winning model | RMSE | DM-validated ────────────
report_table = backtest_summary[['Variable', 'Winning model', 'RMSE', 'DM-validated']]
print(f'Report table  (RMSE = recursive OOS {BACKTEST_H}-step; '
      f'DM-validated = beats naive RW under Clark-West):\n')
print(report_table.to_string(index=False))

# ── Diagnostics (appendix only — not the report table) ──────────────────────────
print('\nDiagnostics (skill vs naive %, HLN-DM, Clark-West, #origins):')
diag = ['Variable', '_skill', '_r2_oos', '_mae', '_mae_naive',
        '_dm_star', '_dm_p', '_cw_stat', '_cw_p', '_lb_stat', '_lb_p', '_n_origins']
diag = [c for c in diag if c in backtest_summary.columns]
print(backtest_summary[diag].rename(columns=lambda c: c.lstrip('_')).to_string(index=False))

# ── Derived-YoY head-to-head (levels-first: reconstruct YoY from the parent level) ──
# The YoY regressors feed Stage B, so the TS-vs-ML head-to-head for them is on the YoY
# series. To keep the levels-first design coherent, the YoY forecast is DERIVED from
# the parent level's OOS forecast rather than modelled independently:
#     YoY_t = L_t / L_{t-4} - 1      =>      yoy_error_t = level_error_t / L_{t-4}
# i.e. the derived-YoY errors are the stored level errors rescaled by the (historical)
# year-ago base. Evaluation starts 2023 Q1 to exclude COVID-distorted year-ago bases —
# an EX-ANTE economic exclusion, so state it as a pre-registered rule in the write-up.
# Caveat: oil's 2022 base is the energy-crisis spike (a genuine signal, not a COVID
# artefact), so us_oil_yoy from 2023 is graded against that elevated base.
YOY_PARENTS = {out: srccol for out, (srccol, meth, per) in DERIVED_VARS.items()
               if meth == 'pct_change_yoy'}
_yoy_start = pd.Timestamp(YOY_EVAL_START)
yoy_rows, yoy_errors = [], {}
for yoy_name, parent in YOY_PARENTS.items():
    if parent not in bt_errors:
        continue
    d    = bt_errors[parent]
    base = df_hist[parent].reindex(d['dates'] - pd.DateOffset(years=1)).values   # L_{t-4}
    keep = (d['dates'] >= _yoy_start) & np.isfinite(base) & (base != 0)
    if keep.sum() < 3:
        yoy_rows.append({'Variable': yoy_name, 'Parent': parent, 'RMSE': np.nan,
                         'DM-validated': 'n/a', '_n_origins': int(keep.sum()),
                         '_validated': False})
        continue
    b    = base[keep]
    ey_m = d['e_model'][keep] / b            # derived-YoY errors (actual - forecast)
    ey_n = d['e_naive'][keep] / b
    fy_m = d['f_model'][keep] / b - 1        # derived-YoY forecasts (model / naive)
    fy_n = d['f_naive'][keep] / b - 1
    od_y = d['dates'][keep]
    rmse, rmse_n = float(np.sqrt(np.mean(ey_m ** 2))), float(np.sqrt(np.mean(ey_n ** 2)))
    dm, p_dm = dm_hln(ey_n, ey_m)
    cw, p_cw = clark_west(ey_n, ey_m, fy_n, fy_m)
    mae, mae_n = float(np.mean(np.abs(ey_m))), float(np.mean(np.abs(ey_n)))
    sse_n  = float(np.sum(ey_n ** 2))
    r2_oos = float(1 - np.sum(ey_m ** 2) / sse_n) if sse_n > 0 else np.nan
    lbq, lbq_p, lbq_lag = ljung_box_oos(ey_m)
    beats = rmse < rmse_n
    validated = bool(beats and np.isfinite(p_cw) and p_cw < DM_ALPHA)
    yoy_rows.append({
        'Variable': yoy_name, 'Parent': parent,
        'RMSE': round(rmse, 5), 'DM-validated': 'Yes' if validated else 'No',
        '_skill': round(100 * (1 - rmse / rmse_n), 1) if rmse_n else np.nan,
        '_r2_oos':  round(r2_oos, 3) if np.isfinite(r2_oos) else None,
        '_mae':     round(mae, 5), '_mae_naive': round(mae_n, 5),
        '_dm_star': round(dm, 3) if np.isfinite(dm) else None,
        '_dm_p':   round(p_dm, 4) if np.isfinite(p_dm) else None,
        '_cw_stat': round(cw, 3) if np.isfinite(cw) else None,
        '_cw_p':   round(p_cw, 4) if np.isfinite(p_cw) else None,
        '_lb_lag':  lbq_lag,
        '_lb_stat': round(lbq, 3) if np.isfinite(lbq) else None,
        '_lb_p':    round(lbq_p, 4) if np.isfinite(lbq_p) else None,
        '_n_origins': int(keep.sum()), '_validated': validated,
    })
    yoy_errors[yoy_name] = dict(e_model=ey_m, e_naive=ey_n, dates=od_y, parent=parent)

yoy_summary = pd.DataFrame(yoy_rows)
print(f'\nDerived-YoY head-to-head  (YoY reconstructed from parent-level OOS forecasts; '
      f'eval from {YOY_EVAL_START}):')
if not yoy_summary.empty:
    ycols = ['Variable', 'Parent', 'RMSE', 'DM-validated', '_skill', '_r2_oos', '_mae',
             '_dm_star', '_dm_p', '_cw_stat', '_cw_p', '_lb_stat', '_lb_p', '_n_origins']
    ycols = [c for c in ycols if c in yoy_summary.columns]
    print(yoy_summary[ycols].rename(columns=lambda c: c.lstrip('_')).to_string(index=False))
    _nmax = int(yoy_summary['_n_origins'].max())
    print(f'  NOTE: only ~{_nmax} post-{_yoy_start.year} origins — DM/CW are very low-powered '
          f'here; read "DM-validated = No" as "not shown to beat naive", not "equal to naive".')

# ── Export comparable metrics + aligned OOS errors (for the TS-vs-ML head-to-head) ─
# The ML strand reports the same columns and loads the error file below, merges on
# (Variable, date), then runs the IDENTICAL dm_hln on the two error samples. Note
# dm_hln(e_a, e_b) uses d = e_a**2 - e_b**2, so d>0 => the SECOND argument wins;
# both strands must use h=1 and the merge on date enforces common evaluation points.
try:
    LOCAL_OUTPUT.mkdir(parents=True, exist_ok=True)
except Exception:
    pass

cmp_cols = ['Variable', 'Winning model', 'RMSE', '_mae', '_r2_oos',
            '_dm_star', '_dm_p', '_cw_stat', '_cw_p', '_lb_stat', '_lb_p',
            '_n_origins', 'DM-validated']
cmp_cols = [c for c in cmp_cols if c in backtest_summary.columns]
cmp_table = backtest_summary[cmp_cols].rename(columns=lambda c: c.lstrip('_'))
cmp_path = LOCAL_OUTPUT / f'{MODEL_PREFIX}_oos_metrics_US.csv'
try:
    cmp_table.to_csv(cmp_path, index=False); print(f'\nComparable OOS metrics -> {cmp_path}')
except Exception as e:
    print(f'\n(metrics CSV not written: {e})')

err_frames = []
for _v, _d in bt_errors.items():
    err_frames.append(pd.DataFrame({
        'Variable': _v, 'date': _d['dates'],
        f'{MODEL_PREFIX}_error': _d['e_model'], 'naive_error': _d['e_naive']}))
if err_frames:
    err_long = pd.concat(err_frames, ignore_index=True)
    err_path = LOCAL_OUTPUT / f'{MODEL_PREFIX}_oos_errors_US.csv'
    try:
        err_long.to_csv(err_path, index=False)
        print(f'OOS error samples   -> {err_path}  ({len(err_long)} rows)')
    except Exception as e:
        print(f'(error CSV not written: {e})')

# ── Derived-YoY head-to-head exports (parallel to the level exports above) ─────────
# ML strand: export the same, load the error file, merge on (Variable, date), and run
# the identical dm_hln on the two YoY error columns. Both strands must evaluate the
# SAME post-2023 targets, so the merge on date enforces the common window.
if not yoy_summary.empty:
    ycmp_cols = ['Variable', 'Parent', 'RMSE', '_mae', '_r2_oos', '_dm_star', '_dm_p',
                 '_cw_stat', '_cw_p', '_lb_stat', '_lb_p', '_n_origins', 'DM-validated']
    ycmp_cols = [c for c in ycmp_cols if c in yoy_summary.columns]
    ycmp = yoy_summary[ycmp_cols].rename(columns=lambda c: c.lstrip('_'))
    ymet_path = LOCAL_OUTPUT / f'{MODEL_PREFIX}_yoy_oos_metrics_US.csv'
    try:
        ycmp.to_csv(ymet_path, index=False); print(f'\nDerived-YoY metrics -> {ymet_path}')
    except Exception as e:
        print(f'\n(YoY metrics CSV not written: {e})')
    yerr_frames = []
    for _v, _d in yoy_errors.items():
        yerr_frames.append(pd.DataFrame({
            'Variable': _v, 'Parent': _d['parent'], 'date': _d['dates'],
            f'{MODEL_PREFIX}_error': _d['e_model'], 'naive_error': _d['e_naive']}))
    if yerr_frames:
        yerr_long = pd.concat(yerr_frames, ignore_index=True)
        yerr_path = LOCAL_OUTPUT / f'{MODEL_PREFIX}_yoy_oos_errors_US.csv'
        try:
            yerr_long.to_csv(yerr_path, index=False)
            print(f'Derived-YoY errors  -> {yerr_path}  ({len(yerr_long)} rows)')
        except Exception as e:
            print(f'(YoY error CSV not written: {e})')

# ── booktabs LaTeX export ───────────────────────────────────────────────────────
def _tex_escape(s): return str(s).replace('_', r'\_')
def report_to_latex(df):
    out = [r'\begin{tabular}{llrc}', r'  \toprule',
           r'  Variable & Winning model & RMSE & DM-validated \\', r'  \midrule']
    for _, r in df.iterrows():
        rmse = '--' if pd.isna(r['RMSE']) else f"${r['RMSE']}$"
        out.append('  {} & {} & {} & {} \\\\'.format(
            _tex_escape(r['Variable']), r['Winning model'], rmse, r['DM-validated']))
    out += [r'  \bottomrule', r'\end{tabular}']
    return '\n'.join(out)

latex_table = report_to_latex(report_table)
tex_path = LOCAL_OUTPUT / f'{MODEL_PREFIX}_backtest_summary_US.tex'
try:
    tex_path.write_text(latex_table); print(f'\nLaTeX table -> {tex_path}')
except Exception as e:
    print(f'\n(LaTeX not written to disk: {e})')
print('\n' + latex_table)


In [ ]:
# ── Figure 3 — ALL forecasts (annotated with winning model, OOS RMSE, DM flag) ──
def _panel_title(var):
    row = backtest_summary.loc[backtest_summary['Variable'] == var]
    base = var.replace('us_', '').replace('_', ' ').title()
    if row.empty:
        return base
    r = row.iloc[0]
    mark = '\u2713' if (r.get('_validated') == True) else '\u2717'
    rmse = '' if pd.isna(r['RMSE']) else f" | RMSE {r['RMSE']:g}"
    return f"{base}<br><sub>{r['Winning model']}{rmse} | DM {mark}</sub>"

def plot_forecast_panels(var_list, fig_title):
    if not var_list:
        print('  (no variables to plot)'); return None
    n_cols = 3
    n_rows = int(np.ceil(len(var_list) / n_cols))
    fig = make_subplots(rows=n_rows, cols=n_cols,
                        subplot_titles=[_panel_title(v) for v in var_list],
                        vertical_spacing=0.11, horizontal_spacing=0.06)
    for i, var in enumerate(var_list):
        r, c = i // n_cols + 1, i % n_cols + 1
        hist = df_hist[var].dropna()
        fc   = forecast_raw[var] if var in forecast_raw.columns else None
        fig.add_trace(go.Scatter(x=hist.index[-40:], y=hist.values[-40:], mode='lines',
                                 line=dict(color=NAVY, width=1.5),
                                 showlegend=(i == 0), name='Historical'), row=r, col=c)
        if fc is not None:
            fig.add_trace(go.Scatter(x=fc.index, y=fc.values, mode='lines',
                                     line=dict(color=RED, width=1.5, dash='dash'),
                                     showlegend=(i == 0), name='SARIMA Forecast'), row=r, col=c)
    fig.update_layout(title=dict(text=fig_title, font=dict(size=13, color=NAVY)),
                      template=TEMPLATE, height=n_rows * 260,
                      margin=dict(t=95, b=40, l=40, r=20))
    fig.update_annotations(font_size=9)
    fig.show()
    return fig

fig_all = plot_forecast_panels(
    [v for v in RAW_TO_FORECAST if v in forecast_raw.columns],
    'Figure 3 \u2014 All SARIMA forecasts (2026\u20132030)<br>'
    '<sup>Per panel: winning model | recursive out-of-sample RMSE | '
    'DM-validated \u2713/\u2717 vs naive random walk</sup>')


In [ ]:
# ── Figure 4 — DM-VALIDATED forecasts only (significantly beat a naive RW) ──────
validated_vars = [r['Variable'] for _, r in backtest_summary.iterrows()
                  if (r.get('_validated') == True) and r['Variable'] in forecast_raw.columns]
not_validated  = [r['Variable'] for _, r in backtest_summary.iterrows()
                  if (r.get('_validated') != True) and r['Variable'] in forecast_raw.columns]

print(f'DM-validated ({len(validated_vars)}): {validated_vars}')
print(f'Not validated ({len(not_validated)}): {not_validated}\n')

fig_validated = plot_forecast_panels(
    validated_vars,
    'Figure 4 \u2014 DM-validated forecasts only (2026\u20132030)<br>'
    '<sup>Only series whose winning model significantly beats a naive random walk '
    '(Clark-West, one-sided, 5%). Series not shown here are not demonstrably '
    'better than a no-change forecast.</sup>')
